<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week06-rag-foundations/Nugget032_Relevance_Thresholding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 026: Persistent Investigation Memory

In [36]:
!pip install -q google-genai
!pip install sentence-transformers
import json
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [38]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [39]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [40]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  resJsonObj  = responseObj[objname]
  return resJsonObj



In [41]:
def save_investigation(memory, investigation):

    memory.append(investigation)

    return memory

In [42]:
def get_last_investigation(memory):

    if len(memory) == 0:
        return None

    return memory[-1]

Load investigation history

Copy data from github project data/investigation_memory.json

In [91]:
data=[
  {
    "investigation_id": 1,
    "date": "2026-06-19",
    "dormant_accounts": 70,
    "inactive_approvers": 35,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH",
    "recommendation": "Enable approver monitoring"
  },
  {
    "investigation_id": 2,
    "date": "2026-06-20",
    "dormant_accounts": 75,
    "inactive_approvers": 25,
    "sla_compliance": 70,
    "root_cause": "Workflow Bottleneck",
    "confidence": "HIGH",
    "recommendation": "Optimize workflow routing"
  },
  {
    "investigation_id": 3,
    "date": "2026-06-21",
    "dormant_accounts": 65,
    "inactive_approvers": 25,
    "sla_compliance": 35,
    "root_cause": "Provisioning Delays",
    "confidence": "HIGH",
    "recommendation": "Increase connector capacity"
  },
  {
    "investigation_id": 4,
    "date": "2026-06-22",
    "dormant_accounts": 85,
    "inactive_approvers": 15,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause":"Approval Queue Growth",
    "recommendation":"Reduce approval backlog"
  },
  {
    "investigation_id": 5,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "confidence": "HIGH",
    "root_cause":"Escalation Failure",
    "recommendation":"Review escalation rules"
  }
]

Load data to current notebook context

In [92]:
with open("investigation_memory.json", "w") as f:
    json.dump(data, f)

Load investigation history now

In [93]:
with open("investigation_memory.json") as f:
    investigations = json.load(f)

print(investigations)

[{'investigation_id': 1, 'date': '2026-06-19', 'dormant_accounts': 70, 'inactive_approvers': 35, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH', 'recommendation': 'Enable approver monitoring'}, {'investigation_id': 2, 'date': '2026-06-20', 'dormant_accounts': 75, 'inactive_approvers': 25, 'sla_compliance': 70, 'root_cause': 'Workflow Bottleneck', 'confidence': 'HIGH', 'recommendation': 'Optimize workflow routing'}, {'investigation_id': 3, 'date': '2026-06-21', 'dormant_accounts': 65, 'inactive_approvers': 25, 'sla_compliance': 35, 'root_cause': 'Provisioning Delays', 'confidence': 'HIGH', 'recommendation': 'Increase connector capacity'}, {'investigation_id': 4, 'date': '2026-06-22', 'dormant_accounts': 85, 'inactive_approvers': 15, 'sla_compliance': 72, 'confidence': 'HIGH', 'root_cause': 'Approval Queue Growth', 'recommendation': 'Reduce approval backlog'}, {'investigation_id': 5, 'date': '2026-06-23', 'dormant_accounts': 115, 'inactive_approvers': 115,

Create searchable text.

In [96]:
documents = []

for inv in investigations:

    documents.append(
        f"""
        Root Cause:
        {inv['root_cause']}

        Recommendation:
        {inv['recommendation']}
        """
    )

Generate Embeddings

In [97]:
embeddings = model.encode(documents)

Create retrieval function and test it

In [98]:
def retrieve(query):

    query_embedding = model.encode(query)

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    best_index = scores.argmax()

    return documents[best_index]

In [120]:
def retrieve_top3(query):

    query_embedding = model.encode(query)

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    top_indices = scores[0].argsort(descending=True)[:3]

    contexts = []

    for idx in top_indices:
        contexts.append(
            f"""
            {documents[idx]}

            score:
            {float(scores[0][idx])}
            """
        )

    context = "\n\n".join(contexts)
    return context

Retrieve Relevent docs based on threshold

In [140]:
def retrieve_top3_relevent(query,THRESHOLD):

    query_embedding = model.encode(query)

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    for i, score in enumerate(scores[0]):
      print(f"{i}: {float(score):.4f}")

    top_indices = scores[0].argsort(descending=True)[:3]
    #print(top_indices)

    relevant_docs = []

    for idx in top_indices:

        similarity = float(
            scores[0][idx]
        )

        if similarity >= THRESHOLD:

            relevant_docs.append(
              f"""
              {documents[idx]}
        score:
        {float(scores[0][idx])}
        """
            )



    context = "\n\n".join(relevant_docs)
    return context

In [141]:
context = retrieve_top3_relevent(
    "Certification campaigns are delayed.",
    0.21
)

print(context)

0: 0.2361
1: 0.1144
2: 0.3497
3: 0.2057
4: 0.1912

              
        Root Cause:
        Provisioning Delays

        Recommendation:
        Increase connector capacity
        
        score:
        0.34968721866607666
        


              
        Root Cause:
        Inactive Approvers

        Recommendation:
        Enable approver monitoring
        
        score:
        0.23605018854141235
        


Build a RAG now

In [142]:
query = """
Certification campaigns are delayed.
"""

In [143]:
context = retrieve_top3_relevent(query,0.21)

0: 0.2361
1: 0.1144
2: 0.3497
3: 0.2057
4: 0.1912


Prompt LLM

In [144]:
prompt = f"""
You are an IAM investigation assistant.

Question:

{query}

Relevant Historical Investigations:

{context}

Generate:

1. Executive Summary
2. Likely Causes
3. Recommended Actions
"""

In [145]:
print(callGPT(prompt).text)

Here's an analysis of the delayed certification campaigns:

---

### 1. Executive Summary

Certification campaigns are currently experiencing significant delays, posing a critical risk to the organization's compliance posture and overall security. Untimely access reviews can lead to prolonged unauthorized access, potential audit findings, and increased security vulnerabilities. Initial assessment, informed by historical investigation data, suggests that core issues are likely related to system capacity for data operations and the timely participation of designated approvers in the review process. Addressing these areas is paramount to restoring the efficiency and effectiveness of our access certification program.

### 2. Likely Causes

Based on historical investigations and the provided scores, the most likely causes for the delayed certification campaigns are:

1.  **Provisioning Delays (High Likelihood):**
    *   **Reasoning:** This is identified as the most probable root cause (sco